# Módulo 3 - Punto 3.2: Procesamiento en video y curva temporal fisiológica

Toda la lógica del cálculo está en `src/modulo3/curva_temporal.py`; este notebook se encarga de llamar a los módulos y visualizar los resultados.

Consigna: (1) segmentar y medir área/diámetro **frame a frame** y validar contra la medición manual (MAE/RMSE); (2) graficar A(t) o D(t); (3) extraer latencia, amplitud, velocidad de constricción y de redilatación, y reportar la variabilidad entre trials; (4) documentar las fuentes de artefactos y el método para filtrarlos o interpolarlos.

> **Aviso de privacidad (Art. 6 del Protocolo — privacidad por diseño).** El video y los frames del ojo son datos biométricos de personas identificables (el `config.yaml` nombra `Video_Sofi_1` y `Frame_Delca_1`). Antes de cargarlos en cualquier sistema de IA externo hay que contar con consentimiento, recortar solo la ROI y sacar nombres de archivo, y recordar que un sistema externo implica **transferencia internacional de datos**. Correr este notebook de forma local no envía nada a terceros.

In [ ]:
# --- Imports de la librería estándar ---
import sys
from pathlib import Path

# --- Configurar la raíz del repositorio ---
# Path.cwd() = carpeta desde donde corre el notebook (normalmente notebooks/).
# Subimos carpetas hasta encontrar config.yaml: esa es la raíz.
raiz = Path.cwd()
while not (raiz / "config.yaml").exists():
    if raiz.parent == raiz:          # llegamos a "/" sin encontrarlo
        raise FileNotFoundError("No encontré config.yaml en ninguna carpeta superior.")
    raiz = raiz.parent

# Insertamos la raíz al principio de sys.path (posición 0 = máxima prioridad).
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

# --- Imports de terceros y propios
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import cargar_config, ruta_absoluta
from src.modulo2 import anotacion as an
from src.modulo3 import curva_temporal as ct

## 1. Carga de Datos y Configuración Global
Se leen los parámetros desde `config.yaml` (bloque `modulo3_2`), se definen las rutas de salida y se decide el modo de trabajo: **modo video** si el archivo existe; si no, **modo CSV** con la serie ya medida en el Módulo 4.

Los valores por defecto de abajo son **míos, no validados sobre este video**: hay que declararlos como parámetros en el informe y ajustarlos mirando los gráficos. El instante del estímulo (`estimulos_s`) tiene que salir del protocolo experimental; **no se infiere de la curva**.

In [ ]:
# Cargar configuración global
cfg = cargar_config(raiz / "config.yaml")

# Directorio de salida de resultados
carpeta_res = raiz / "results" / "modulo3"
carpeta_res.mkdir(parents=True, exist_ok=True)

# Rutas de entrada
ruta_video = ruta_absoluta(cfg["entrada"]["video_path"])
carpeta_anot = raiz / "results" / "modulo2" / "anotaciones_manuales"
csv_m4 = raiz / "results" / "modulo4" / "4_diametro_video.csv"

# Parámetros del módulo 3.2 (si falta el bloque en config.yaml se usan estos por defecto)
p = {"paso_frames": 1, "estimulos_s": [], "ventana_basal_s": 1.0, "ventana_respuesta_s": 3.0,
     "max_hueco_s": 0.5, "sg_ventana_s": 0.3, "sg_orden": 2}
p.update(cfg.get("modulo3_2", {}))
paso_frames = p["paso_frames"]
estimulos_s = list(p["estimulos_s"])     # COMPLETAR en config.yaml con el protocolo

usar_video = ruta_video.exists()
print(f"Resultados en: {carpeta_res}")
print("Modo         :", "VIDEO" if usar_video else f"CSV (no encuentro {ruta_video.name})")
print("Estímulos    :", estimulos_s if estimulos_s else "SIN DEFINIR (se saltean los parámetros fisiológicos)")

## 2. Segmentación frame a frame

`ct.serie_temporal` recorre el video y, en cada frame, recorta la ROI, segmenta con **Watershed** (`seg.segmentar_pupila`, Módulo 2), aplica la operación morfológica final si está configurada y calcula con `descriptores_forma` (Módulo 3.1) el área, la circularidad, la excentricidad y el centroide. El diámetro es el **equivalente** D = 2·√(A/π), igual que en el Módulo 4.

Resolución temporal = 1/fs: con `paso_frames = 3` es ~0,1 s; con `paso_frames = 1`, ~0,033 s. En **modo CSV** se usa la serie `original` de `4_diametro_video.csv` (solo área y diámetro, sin forma ni centroide).

In [ ]:
if usar_video:
    serie = ct.serie_temporal(ruta_video, cfg, paso_frames=paso_frames)
    serie.to_csv(carpeta_res / "3_2_serie_cruda.csv", index=False)
else:
    m4 = pd.read_csv(csv_m4)
    original = m4[m4["version"] == "original"].reset_index(drop=True)
    serie = original.assign(frame=np.arange(len(original)), motivo="ok")
    serie.attrs["fs_hz"] = 1.0 / np.median(np.diff(serie["t_s"]))

fs = serie.attrs["fs_hz"]
print(f"Muestras: {len(serie)} | fs efectiva: {fs:.2f} Hz | duración cubierta: {serie['t_s'].max():.2f} s")
serie.head()

## 3. Validación contra la medición manual (MAE / RMSE)

Se miden automáticamente los frames anotados a mano (`results/modulo2/anotaciones_manuales`) y se comparan área y diámetro. **MAE** = media de |auto − manual|; **RMSE** = raíz de la media de (auto − manual)²; **sesgo** = media de (auto − manual) (positivo = el automático sobreestima).

Se informan dos subconjuntos para no esconder nada: *todos* (una falla grosera domina el RMSE) y uno sin fallas groseras (en modo video: Dice ≥ 0,5; en modo CSV: D_auto ≤ 1,5·D_manual — criterios a justificar). Con pocos frames anotados las cifras son orientativas.

En **modo CSV** la comparación es aproximada: se usa el frame de la serie más cercano en el tiempo a cada anotación (desfase de hasta medio paso de muestreo).

In [ ]:
if usar_video:
    tabla_val, resumen_val = ct.validar_contra_manual(ruta_video, cfg, carpeta_anot)
else:
    manuales = an.cargar_anotaciones(carpeta_anot)
    tabla_val, resumen_val = ct.validar_contra_serie(manuales, serie)

if tabla_val is None:
    print("No hay anotaciones manuales para validar.")
else:
    tabla_val.to_csv(carpeta_res / "3_2_validacion_por_frame.csv", index=False)
    resumen_val.to_csv(carpeta_res / "3_2_validacion_resumen.csv", index=False)
    display(tabla_val.round(2))
    display(resumen_val.round(2))

## 4. Artefactos: fuentes y método de detección

| Fuente | Qué pasa en la imagen | Cómo se detecta (`ct.detectar_artefactos`) |
|---|---|---|
| **Parpadeo completo** | El párpado tapa el iris | `estimar_pupila` devuelve `None` (contraste del iris < `contraste_min`): fila con NaN, motivo `sin_iris` |
| **Parpadeo parcial / pestañas / párpado sobre la pupila** | La máscara deja de parecerse a una elipse y el área da un salto | Caen la circularidad y sube la excentricidad (puntaje robusto con mediana/MAD, umbral `k_forma`) |
| **Movimiento ocular** (sacada, deriva) | El centroide salta respecto de su trayectoria; la pupila se ve más ovalada y el área proyectada baja sin que cambie el tamaño real | Salto del centroide respecto de su mediana móvil (umbral `k_centro`, piso de 1 px) |
| **Atípicos puntuales** (reflejos, sombras, error de Watershed) | Picos aislados de área | Filtro de **Hampel** sobre log(área): se marca lo que se aleja más de `k_hampel`·1,4826·MAD de la mediana móvil |

Dos detalles de diseño:

- **Protección contra constricciones reales:** un tramo marcado por Hampel solo queda como artefacto si la señal *vuelve al nivel de antes* (un pico). Si queda en otro nivel (un escalón), es fisiología y se desmarca. Sin esto, el filtro marcaba como atípica parte de la caída real de ~4,7 s.
- **Dilatación:** cada marca se extiende `dilatar_s` = 0,1 s a cada lado, porque los frames vecinos a un parpadeo suelen estar contaminados.

**Qué pasa con lo marcado** (`ct.limpiar_serie`, sección 5): se sacan; los **huecos cortos** (≤ `max_hueco_s`) se interpolan linealmente; los **huecos largos quedan en NaN** (no se inventan datos); el suavizado Savitzky-Golay se hace tramo por tramo. Los umbrales son valores por defecto míos, no validados.

In [ ]:
serie_a = ct.detectar_artefactos(serie)

criterios = ["art_sin_seg", "art_forma", "art_centro", "art_atipico"]
print("Frames marcados por criterio (antes de dilatar):")
print(serie_a[criterios].sum().to_string())
print(f"\nFrames descartados en total (con dilatación): {int(serie_a['artefacto'].sum())} de {len(serie_a)}")
if not usar_video:
    print("(Modo CSV: solo actúa el criterio de área; no hay circularidad ni centroide.)")

serie_a.loc[serie_a["artefacto"], ["t_s", "area_px", "diametro_px"] + criterios].round(2)

## 5. Limpieza: descartar, interpolar huecos cortos y suavizar

In [ ]:
serie_l = ct.limpiar_serie(serie_a, max_hueco_s=p["max_hueco_s"],
                           sg_ventana_s=p["sg_ventana_s"], sg_orden=p["sg_orden"])
serie_l.to_csv(carpeta_res / "3_2_serie_temporal.csv", index=False)

n_nan = int(serie_l["area_px_suave"].isna().sum())
print(f"Muestras que quedaron en NaN (huecos largos, no interpolados): {n_nan}")
if n_nan:
    print("En t =", np.round(serie_l["t_s"][serie_l["area_px_suave"].isna()].values, 2).tolist(), "s")

## 6. Curva temporal A(t) y D(t)

Gris: medición cruda. Cruces rojas: frames descartados. Azul: serie limpia. Negro: suavizada. Líneas naranjas: estímulos (si están definidos).

In [ ]:
art = serie_l["artefacto"].to_numpy()
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

# --- Área ---
ax = axes[0]
ax.plot(serie_l["t_s"], serie_l["area_px"], ".", color="0.65", ms=4, label="medición cruda")
ax.plot(serie_l["t_s"][art], serie_l["area_px"][art], "x", color="tab:red", ms=6, label="descartada (artefacto)")
ax.plot(serie_l["t_s"], serie_l["area_px_limpia"], "-", color="tab:blue", lw=0.8, alpha=0.6, label="limpia")
ax.plot(serie_l["t_s"], serie_l["area_px_suave"], "-", color="k", lw=2, label="suavizada (Savitzky-Golay)")
ax.set_ylabel("Área pupilar [px²]")
ax.set_title("Área pupilar A(t)", fontsize=12, fontweight="bold")

# --- Diámetro equivalente ---
ax = axes[1]
d_cruda = ct.diametro_desde_area(serie_l["area_px"])
ax.plot(serie_l["t_s"], d_cruda, ".", color="0.65", ms=4, label="cruda")
ax.plot(serie_l["t_s"][art], d_cruda[art], "x", color="tab:red", ms=6, label="descartada")
ax.plot(serie_l["t_s"], ct.diametro_desde_area(serie_l["area_px_suave"]), "k-", lw=2, label="suavizada")
ax.set_ylabel("Diámetro equivalente [px]")
ax.set_xlabel("Tiempo [s]")
ax.set_title("Diámetro pupilar D(t)", fontsize=12, fontweight="bold")

for ax in axes:
    for i, te in enumerate(estimulos_s):
        ax.axvline(te, color="tab:orange", ls="--", label="estímulo" if i == 0 else None)
    ax.grid(alpha=0.3)
    ax.margins(y=0.25)                       # espacio arriba para la leyenda
    ax.legend(ncol=4, fontsize=8, loc="upper center")

plt.tight_layout()
fig.savefig(carpeta_res / "3_2_curva_temporal.png", dpi=150)
plt.show()

## 7. Parámetros fisiológicos del reflejo fotomotor

Para cada estímulo (`ct.parametros_plr`):

| Parámetro | Definición |
|---|---|
| **Área basal** | Mediana del área suavizada en `[t_est − ventana_basal, t_est)`. Tiene que ser el tramo en oscuridad. |
| **Mínimo** | Mínimo del área suavizada en `[t_est, t_est + ventana_respuesta]` (o hasta el estímulo siguiente). |
| **Amplitud** | basal − mínimo (px²), y en % del basal. |
| **Latencia (umbral)** | Primer instante ≥ t_est en que la curva baja del umbral `basal − max(3·σ_basal, 3 % del basal)` durante 3 muestras seguidas (con interpolación en el cruce), menos t_est. σ_basal = 1,4826·MAD del tramo basal. |
| **Latencia (tangente)** | Instante en que la tangente en el punto de máxima velocidad de constricción corta el nivel basal, menos t_est. Menos sensible al umbral; si difiere mucho de la anterior, mirar el gráfico. |
| **Vel. de constricción** | Máxima (pico de −dA/dt entre el inicio y el mínimo) y media (amplitud / (t_mínimo − t_inicio)). |
| **Vel. de redilatación** | Máxima (pico de dA/dt tras el mínimo) y media entre el mínimo y el 75 % de recuperación de la amplitud (NaN si no se recupera dentro del registro). |

Cada fila trae `frac_artefactos_ventana` (qué fracción de la ventana de respuesta estaba contaminada), `resolucion_temporal_s` y una `advertencia` si algo no se pudo calcular.

In [ ]:
if estimulos_s:
    tabla_plr = ct.parametros_todos(serie_l, estimulos_s,
                                    ventana_basal_s=p["ventana_basal_s"],
                                    ventana_respuesta_s=p["ventana_respuesta_s"])
    tabla_plr.to_csv(carpeta_res / "3_2_parametros_plr.csv", index=False)
    display(tabla_plr.T)

    # Verificación visual: basal (verde), mínimo (triángulo) e inicio de constricción (punteada)
    fig, ax = plt.subplots(figsize=(11, 4.2))
    ax.plot(serie_l["t_s"], serie_l["area_px_suave"], "k-", lw=2, label="suavizada")
    for _, r in tabla_plr.iterrows():
        ax.axvline(r["t_estimulo_s"], color="tab:orange", ls="--")
        ax.hlines(r["basal"], r["t_estimulo_s"] - p["ventana_basal_s"], r["t_estimulo_s"], color="tab:green", lw=3)
        ax.plot(r["t_minimo_s"], r["minimo"], "v", color="tab:purple", ms=9)
        if np.isfinite(r["latencia_s"]):
            ax.axvline(r["t_estimulo_s"] + r["latencia_s"], color="tab:green", ls=":")
    ax.set_xlabel("Tiempo [s]"); ax.set_ylabel("Área pupilar [px²]")
    ax.set_title("Parámetros sobre la curva", fontsize=12, fontweight="bold")
    ax.grid(alpha=0.3); ax.legend()
    plt.tight_layout()
    fig.savefig(carpeta_res / "3_2_parametros_sobre_curva.png", dpi=150)
    plt.show()
else:
    tabla_plr = None
    print("Definí `estimulos_s` en config.yaml (modulo3_2) para calcular latencia, amplitud y velocidades.")

## 8. Variabilidad entre trials / registros

`ct.resumen_trials` informa por parámetro: n, media, desvío (ddof = 1) y coeficiente de variación (%). Con un solo trial el desvío no se puede calcular (NaN). Si los trials son **registros distintos** (varios videos), corré las secciones 2–7 para cada uno y concatená las tablas con `pd.concat([...], ignore_index=True)` antes de llamar a `resumen_trials`.

In [ ]:
if tabla_plr is not None:
    resumen_tr = ct.resumen_trials(tabla_plr)
    resumen_tr.to_csv(carpeta_res / "3_2_variabilidad_trials.csv")
    display(resumen_tr.round(3))
else:
    print("Sin tabla de parámetros (falta `estimulos_s`).")

## 9. Verificación del método con una curva sintética

Antes de confiar en los parámetros sobre datos reales se prueba el algoritmo con una respuesta de **verdad conocida** (`ct.plr_sintetica`): basal 300 px², estímulo en 3,0 s, latencia 0,25 s, mínimo 150 px² (amplitud 150) y dos parpadeos inyectados (uno con NaN, otro con salto de área y forma). Esto prueba el algoritmo, **no** los resultados sobre el video real. Al bajar fs la medida se degrada, y por eso se recomienda `paso_frames = 1`.

In [ ]:
filas = []
for fs_s in (30.0, 10.0):
    t, A = ct.plr_sintetica(fs_s)
    rng = np.random.default_rng(1)
    df_s = pd.DataFrame({"t_s": t, "frame": np.arange(len(t)), "area_px": A,
                         "diametro_px": ct.diametro_desde_area(A),
                         "circularidad": 0.9 + rng.normal(0, .01, len(t)),
                         "excentricidad": 0.3 + rng.normal(0, .01, len(t)),
                         "xc": 50 + rng.normal(0, .2, len(t)), "yc": 40 + rng.normal(0, .2, len(t)),
                         "motivo": "ok"})
    df_s.attrs["fs_hz"] = fs_s
    n3 = int(0.3 * fs_s)
    i1, i2 = int(6.0 * fs_s), int(9.0 * fs_s)
    df_s.loc[i1:i1 + n3, ["area_px", "circularidad", "xc", "yc"]] = np.nan   # parpadeo completo
    df_s.loc[i2:i2 + n3, "area_px"] *= 2.3                                     # parpadeo parcial
    df_s.loc[i2:i2 + n3, "circularidad"] = 0.5

    d = ct.limpiar_serie(ct.detectar_artefactos(df_s))
    inyectados = np.zeros(len(t), bool)
    inyectados[i1:i1 + n3 + 1] = True
    inyectados[i2:i2 + n3 + 1] = True
    r = ct.parametros_plr(d, t_estimulo=3.0)
    filas.append(dict(fs_hz=fs_s,
                      cobertura_parpadeos_pct=100 * d["artefacto"][inyectados].mean(),
                      latencia_real_s=0.25, latencia_umbral_s=r["latencia_s"],
                      latencia_tangente_s=r["latencia_tangente_s"],
                      amplitud_real=150.0, amplitud_medida=r["amplitud"]))

pd.DataFrame(filas).round(3)

## Limitaciones y pendientes

- **Falta el video** en el proyecto: `serie_temporal` y `validar_contra_manual` están escritas pero no se corrieron sobre datos reales. Mientras tanto el notebook funciona en modo CSV.
- **Falta el instante del estímulo** y el número de trials: sin ellos no se pueden calcular latencia, amplitud, velocidades ni variabilidad.
- **Duración:** la consigna habla de 30 s y el CSV del Módulo 4 cubre 9,72 s. Hay que confirmarlo con el video.
- Umbrales de artefactos, ventana de respuesta y porcentajes de latencia/recuperación son **valores por defecto, no validados**: declararlos como parámetros y justificarlos con los gráficos.
- **Valores de referencia de la literatura** (latencias o amplitudes típicas): no se citan porque no se consultaron fuentes; si el informe los necesita, hay que buscarlos y citarlos.
- La validación manual se basa en pocos frames (6): MAE/RMSE son orientativos. Revisar el **sesgo positivo** del diámetro automático y las **fallas de segmentación** (picos de ~40 px) antes de interpretar amplitudes; `frac_artefactos_ventana` ayuda a detectar una falla dentro de la ventana de respuesta.